# Disease Early Detection - Bidirectional LSTM with Attention

This notebook demonstrates the training pipeline for early disease detection in livestock using multivariate biometric time-series data.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Input, Attention, Flatten, Activation, RepeatVector, Permute, Multiply, Lambda
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Synthetic Data Generation
def generate_synthetic_data(n_samples=1000, timesteps=288, n_features=3):
    X = np.random.normal(0, 1, (n_samples, timesteps, n_features))
    y = np.random.randint(0, 2, n_samples)
    # Add patterns for 'sick' samples
    X[y == 1, -50:, 0] += 2.0  # Temperature spike
    X[y == 1, -50:, 1] += 1.5  # Heart rate spike
    return X, y

X, y = generate_synthetic_data()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

# 2. Model Architecture
def build_attention_lstm(timesteps=288, n_features=3):
    inputs = Input(shape=(timesteps, n_features))
    lstm_out = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    
    # Attention Mechanism
    attention = Dense(1, activation='tanh')(lstm_out)
    attention = Flatten()(attention)
    attention = Activation('softmax')(attention)
    attention = RepeatVector(128)(attention)
    attention = Permute([2, 1])(attention)
    
    sent_representation = Multiply()([lstm_out, attention])
    sent_representation = Lambda(lambda xin: tf.keras.backend.sum(xin, axis=1))(sent_representation)
    
    x = Dense(32, activation='relu')(sent_representation)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_attention_lstm()
model.summary()

# 3. Training
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32)